# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

### Research Question

Which observable search-performance signals can be combined into a repeatable score to identify content pages that should be prioritized for refresh or review?

### Decision Supported

The analysis is designed to help prioritize which content pages should be reviewed first, based on observed search-performance patterns. The resulting score is a decision-support ranking, not a claim about Google's ranking algorithm or a causal effect of refreshing content.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

### Data

This study uses the FlyRank internship warehouse release on Hugging Face, with the analysis centered on the `fact_content_daily_performance` table.

The table is at daily content-page level: each observation represents a content page for a client on a reporting date. The analysis uses public-safe hashed identifiers and aggregate search-performance measures rather than client names, domains, URLs, private queries, or raw exports.

The primary signals considered are:

- `gsc_impressions` — observed search visibility
- `gsc_clicks` — observed search clicks
- `gsc_avg_position` — observed average search position
- `ga4_pageviews` — observed page traffic where GA4 data is available
- `scroll_events` — observed engagement activity

The analysis excludes sparse AI-referral fields from the baseline opportunity score because these signals contain many zero observations and are not sufficiently dense for the primary analysis.

The final analysis window and row counts are determined from the full warehouse rather than the Hugging Face `first-rows` sample. The sample was used only to inspect schema and field structure.

No client-identifying information, domains, URLs, private queries, credentials, or raw warehouse exports are included in the public analysis.

### Data Availability

Data availability flags are used to distinguish observations where GSC or GA4 measurements are present. Missing or unavailable measurements are not treated as evidence of poor content performance.

### Data Limitations

The warehouse contains observed search and analytics measurements but does not establish why performance changed. It does not by itself identify causal effects of content refreshes, Google algorithm changes, competitor activity, seasonality, or marketing campaigns. Therefore, the resulting opportunity score is framed as directional decision support rather than a causal or algorithmic claim.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

## 3. Methodology

The project uses a repeatable content-opportunity scoring approach. The goal is not to predict Google's ranking system, but to identify pages that deserve earlier human review.

### Features

The analysis uses observable content and search-performance signals including:

- `search_volume`
- `competition`
- `cpc`
- `word_count`
- `impressions_90d`
- `clicks_90d`
- `sessions_90d`
- `content_age_days`
- `ctr`
- `avg_position`
- `engagement_rate`
- `scroll_rate`
- `ai_traffic_pct`
- `trend_pct`

These features represent search demand, competition, content characteristics, visibility, traffic, engagement, and recent movement.

### Opportunity Definition

A content opportunity is treated as a page whose observed signals suggest that it may deserve earlier review or refresh consideration.

There is no direct ground-truth field stating that a page definitely required a refresh. Therefore, the opportunity score is a prioritization tool rather than a verified prediction of refresh success.

### Signal Audit

Before building recommendations, the distributions of the main numeric fields were inspected.

Several features showed strong right-skew and heavy tails. Because extreme observations can strongly affect averages, the analysis uses medians, grouped comparisons, and directional interpretation where appropriate.

Three signals were tested:

1. Trend direction versus recent sessions.
2. Engagement rate versus recent sessions.
3. Search volume versus impressions.

All three were treated as MIXED rather than as guaranteed rules.

### Baseline

The baseline for the classification validation is the majority-class prediction on the same client-grouped test set.

The dataset contained:

- 16,262 declining observations
- 13,738 non-declining observations
- 30,000 observations in total

The majority class represents approximately 54.21% of observations.

### Proposed Model

A Random Forest classifier was evaluated as a stronger modelling approach for identifying declining content.

The model used content, search, traffic, engagement, age, and categorical features while excluding identifiers and target-derived trend fields.

### Validation Design

The original Week-5 random split produced an accuracy of 70.13%.

To test whether that result was optimistic, a client-grouped split was performed using `GroupShuffleSplit`. Records belonging to the same client were kept together rather than allowing the same client to appear in both training and testing groups.

The client-grouped evaluation produced an accuracy of 57.24%.

### Leakage Checks

The target label was created from `trend_direction`.

Therefore, the following target-derived or closely related fields were excluded:

- `trend_direction`
- `trend_pct`
- `impressions_last_30d`
- `clicks_last_30d`
- `sessions_last_30d`
- `impressions_prev_30d`
- `clicks_prev_30d`
- `sessions_prev_30d`

`client_id` and `content_id` were also excluded because they are identifiers rather than useful predictive signals.

The leakage audit therefore reduced the risk of the model directly learning the target definition.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 4. Results (vs baseline)

The final validation was compared against a simple majority-class baseline on the same client-grouped evaluation framework.

### Model vs Baseline

| Approach | Evaluation Metric | Result |
|---|---|---:|
| Majority-class baseline | Accuracy | 54.21% |
| Random Forest — client-grouped split | Accuracy | 57.24% |

The Random Forest measured 57.24% accuracy on the client-grouped test set, compared with 54.21% for the majority-class baseline.

This is a modest improvement of approximately 3.03 percentage points over the majority baseline.

However, the original random split produced 70.13% accuracy. The lower 57.24% result under client-grouped validation indicates that the random-split result likely gives a more optimistic estimate of generalization.

### Random Split vs Grouped Split

| Validation design | Accuracy |
|---|---:|
| Original random split | 70.13% |
| Client-grouped split | 57.24% |

The measured decrease shows why validation design matters. A model can appear substantially stronger when observations from the same client are allowed to appear in both training and testing data.

### Key Findings

- The model performed modestly above the majority-class baseline under client-grouped validation.
- The random-split result was substantially higher than the client-grouped result.
- The grouped result provides a more conservative estimate of performance on unseen clients.
- The tested signals provide directional information that can support content review prioritization.
- The evidence is not strong enough to claim that the model will generalize equally well to every new client or content portfolio.

## 5. Limitations

*What this work cannot claim.*



This analysis has several important limitations.

First, the dataset does not contain a verified ground-truth label for whether a content page actually needed a refresh or whether a refresh successfully improved performance. The declining-content label is therefore a proxy for review prioritization.

Second, several search and traffic features are highly skewed. Extreme observations can influence averages and model relationships, so the results should be interpreted carefully.

Third, the model's accuracy decreased from 70.13% under a random split to 57.24% under a client-grouped split. This indicates that performance on unseen clients may be substantially lower than the original random-split estimate.

Fourth, the analysis measures associations in observed data. It cannot establish that word count, content age, engagement, trend, or another feature causes search-performance changes.

Finally, the recommendations are intended for decision support. Human review is still required before making content, refresh, merge, rewrite, or removal decisions.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*



The analysis supports a review-prioritization workflow rather than an automatic content decision system.

### Priority 1 — Review declining pages

Pages showing declining trend signals should receive early review, especially when recent sessions or other performance indicators also appear weak.

**Reason:** the trend signal showed directional evidence of changing recent performance, but the signal was classified as MIXED.

**Recommended action:** inspect the page before deciding whether to refresh, improve, merge, or monitor it.

### Priority 2 — Review pages with weak engagement

Pages with weaker engagement signals can be reviewed for content relevance, structure, readability, and alignment with user intent.

**Reason:** engagement-rate groups showed directional differences in recent sessions, but the relationship was not strong enough to classify as CONFIRMED.

**Recommended action:** review content quality and user experience before making changes.

### Priority 3 — Review high-demand pages with weak visibility

Pages associated with stronger search demand but weaker observed visibility may represent opportunities for further investigation.

**Reason:** search-volume groups showed directional differences in impressions, although the heavy skew in search volume makes the relationship uncertain.

**Recommended action:** review search intent, content coverage, internal linking, metadata, and current positioning.

### Priority 4 — Review aging content

Older content can be monitored for potential refresh opportunities, especially when age is combined with declining or weak performance signals.

**Reason:** content age is an observable signal, but the analysis cannot establish that age itself causes decline.

**Recommended action:** compare older pages against recent performance before selecting a refresh action.

### Priority 5 — Monitor instead of automatically changing

Pages with mixed or conflicting signals should remain in a monitoring queue rather than receiving an automatic action.

**Reason:** the signal audit found that the tested relationships were directional rather than universally reliable.

**Recommended action:** collect additional observations and use human judgment before making a major content change.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*



The deployed paper should include the following evidence artifacts:

### Artifact 1 — Signal Distribution / Skewness Table

A table showing the main numeric features and their observed skewness.

The strongest observed skew was in:

- `trend_pct` — 56.76
- `search_volume` — 26.02
- `ai_traffic_pct` — 18.43
- `clicks_90d` — 18.35
- `ctr` — 17.44

These values show that several features have heavy right tails.

### Artifact 2 — Signal Test Results

Grouped comparison tables should show:

- trend direction versus recent sessions
- engagement-rate quartiles versus recent sessions
- search-volume quartiles versus impressions

The three signal tests were classified as MIXED.

### Artifact 3 — Validation Comparison

| Validation | Accuracy |
|---|---:|
| Random split | 70.13% |
| Client-grouped split | 57.24% |
| Majority baseline | 54.21% |

This comparison demonstrates the importance of validation design.

### Artifact 4 — Leakage Audit

The final feature set excludes target-derived trend fields, recent/previous 30-day outcome fields, and content/client identifiers.

### Artifact 5 — Ranked Action Playbook

The paper presents the ranked review priorities:

1. Review declining pages.
2. Review weak-engagement pages.
3. Review high-demand / weak-visibility pages.
4. Review aging content.
5. Monitor mixed-signal pages.

# 8. Five-Minute Demo Outline

## Demo Title

Content Opportunity Scoring for Search Performance Review

### 0:00–0:45 — The Question

**Question:**

Which observable search-performance signals can be combined into a repeatable score to identify content pages that should be prioritized for refresh or human review?

**FlyRank content problem:**

Content teams may have many pages to review, but search visibility, traffic, engagement, demand, and content age do not change at the same rate.

The goal is therefore to help reviewers decide **where to look first**, rather than automatically deciding what to change.

---

### 0:45–1:30 — The Data

The analysis uses the FlyRank ML Internship warehouse release, centered on the `fact_content_daily_performance` table.

The analysis considers observable signals including:

- Search impressions
- Search clicks
- Average position
- Traffic
- Engagement
- Search demand
- Content age
- Trend-related signals

Public-safety exclusions were applied. Client-identifying information, domains, URLs, private queries, credentials, and raw warehouse exports were not included in the public research artifact.

---

### 1:30–2:30 — The Method

The workflow was:

1. Define the content-review decision.
2. Inspect the available signals.
3. Build observable predictive features.
4. Define declining vs non-declining observations.
5. Establish a majority-class baseline.
6. Train a Random Forest classifier.
7. Compare random and client-grouped validation.
8. Audit for target leakage.
9. Translate the evidence into a ranked review queue.

The most important validation change was keeping observations from the same client together.

This provides a more conservative test of performance on unseen clients.

---

### 2:30–3:30 — One Chart

## Model vs Baseline

| Approach | Validation | Accuracy |
|---|---|---:|
| Majority-class baseline | Client-grouped | 54.21% |
| Random Forest | Client-grouped | 57.24% |
| Random Forest | Original random split | 70.13% |

The key visual is the comparison between the original random split and the client-grouped result.

The random split produced 70.13% accuracy, while the more conservative client-grouped evaluation produced 57.24%.

This demonstrates why validation design matters when multiple observations may belong to the same client.

---

### 3:30–4:15 — One Honest Result

The Random Forest achieved **57.24% accuracy** under client-grouped validation compared with **54.21%** for the majority-class baseline.

That is an improvement of approximately **3.03 percentage points**.

The result is useful as directional evidence, but it is not strong enough to claim universal generalization or causal impact.

The model does **not** predict Google's ranking algorithm and does not prove that refreshing a page will improve its performance.

---

### 4:15–5:00 — One Recommendation

## Recommendation

**Review declining pages first.**

Pages showing declining trend signals should receive early human review, particularly when other observed performance indicators also appear weak.

The recommended action is not an automatic refresh.

Instead:

1. Inspect the page.
2. Compare recent and historical performance.
3. Check search intent and content quality.
4. Consider refresh, improvement, merge, or monitoring.
5. Keep human judgment in the final decision.

### Closing Message

The main value of this project is not an automated content decision.

It is a repeatable way to reduce the search space and help content teams decide **where to investigate first**.

---

# Shareable Cuts

## 1. Short Social Post

I built a content opportunity scoring framework as part of my FlyRank ML Internship.

The goal was simple: help content teams identify which pages deserve human review first using observable search-performance signals.

I combined search visibility, traffic, engagement, search demand, content age, and trend-related signals, then evaluated a Random Forest using both a conventional random split and a more conservative client-grouped validation design.

The grouped evaluation achieved **57.24% accuracy**, compared with a **54.21% majority-class baseline**.

The biggest lesson was that validation design matters: the original random split reached **70.13%**, but performance dropped when observations from the same client were kept together.

I treated the result as directional decision support rather than a prediction of Google's ranking algorithm or a causal claim about content refreshes.

#MachineLearning #DataScience #SEO #AI #FlyRank #Research

---

## 2. Employer-Facing Summary

I built a content opportunity scoring framework during my FlyRank ML Internship to help prioritize content pages for human review using observable search-performance signals. The analysis used the FlyRank internship warehouse data and combined search visibility, traffic, engagement, search demand, content age, and trend-related features while applying leakage checks and public-safety exclusions. A Random Forest achieved **57.24% accuracy** under client-grouped validation versus a **54.21% majority-class baseline**, demonstrating modest directional value while highlighting the importance of conservative validation and honest interpretation.

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries are included
- [x] Claims use careful words: observed, measured, directional, decision-support
- [x] Validation results are reported honestly
- [x] Leakage checks are documented
- [x] Ranked recommendations are included
- [x] Artifacts for the research paper are identified
- [x] Notebook is committed under `work/notebooks/`
- [ ] Deployed paper URL added to `submission/paper_url.txt`